In [ ]:
!pip install mne tensorflow scikit-learn matplotlib seaborn

In [ ]:
import os
import glob
import numpy as np
import mne
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, cohen_kappa_score, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir('/content/drive/MyDrive')

['Colab Notebooks',
 'BCICIV_2b_gdf.zip',
 'SNS15 Mango Big7 Clubs Roadmap Proof.pdf',
 'SNSCE Mentee Placement Details.gsheet',
 'V.Prabhu - 29.06.2024 & 30.06.2024 - EV Expo 2024-20241214T112737Z-001',
 '1.approj',
 'az_recorder_20241216_142147.mp4',
 'az_recorder_20241216_143012.mp4',
 'video_20241216_151113.mp4',
 'video_20241216_150310.mp4',
 'December 2024-20241217T093827Z-001.zip',
 'December 2024',
 'ATAL FDP Certificate.pdf',
 '23ECB222-DPCO-IAE-Result Analysis.gsheet',
 'Design eFiling Reciept.pdf',
 'Patent Certificates',
 'Product Patent December 2024.gdoc',
 'Flexi Laser Ruler Product Patent December 2024.docx',
 'E-Bike Componets List.gdoc',
 'E-Scooter.gdoc',
 'Untitled spreadsheet (4).gsheet',
 'EV Shuttle Components List.gdoc',
 'Untitled spreadsheet (3).gsheet',
 'Fuji Images',
 'Fuji Hashtag.gdoc',
 'Fuji Lab Inauguration LinkedIn Post.gdoc',
 '1735795950344.jfif',
 'Oracle Academy.pdf',
 'RoadMap for Industry Labs 2025.pdf',
 'Revised 6 pointer Goals_Prabhu.xlsx',
 

In [4]:
os.listdir('/content/drive/MyDrive/BCI_IV_2b')

['B0101T.gdf',
 'B0102T.gdf',
 'B0103T.gdf',
 'B0104E.gdf',
 'B0105E.gdf',
 'B0201T.gdf',
 'B0202T.gdf',
 'B0203T.gdf',
 'B0204E.gdf',
 'B0205E.gdf',
 'B0301T.gdf',
 'B0302T.gdf',
 'B0303T.gdf',
 'B0304E.gdf',
 'B0305E.gdf',
 'B0401T.gdf',
 'B0402T.gdf',
 'B0403T.gdf',
 'B0404E.gdf',
 'B0405E.gdf',
 'B0501T.gdf',
 'B0502T.gdf',
 'B0503T.gdf',
 'B0504E.gdf',
 'B0505E.gdf',
 'B0601T.gdf',
 'B0602T.gdf',
 'B0603T.gdf',
 'B0604E.gdf',
 'B0605E.gdf',
 'B0701T.gdf',
 'B0702T.gdf',
 'B0703T.gdf',
 'B0704E.gdf',
 'B0705E.gdf',
 'B0801T.gdf',
 'B0802T.gdf',
 'B0803T.gdf',
 'B0804E.gdf',
 'B0805E.gdf',
 'B0901T.gdf',
 'B0902T.gdf',
 'B0903T.gdf',
 'B0904E.gdf',
 'B0905E.gdf',
 'BCICIV_2b_gdf.zip']

In [5]:
!unzip "/content/drive/MyDrive/BCI_IV_2b/BCICIV_2b_gdf.zip" -d "/content/drive/MyDrive/BCI_IV_2b"

Archive:  /content/drive/MyDrive/BCI_IV_2b/BCICIV_2b_gdf.zip
replace /content/drive/MyDrive/BCI_IV_2b/B0101T.gdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0101T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0102T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0103T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0104E.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0105E.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0201T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0202T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0203T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0204E.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0205E.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0301T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0302T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0303T.gdf  
  inflating: /content/drive/MyDrive/BCI_IV_2b/B0304E.gdf  
  inflating: /conten

In [ ]:
DATA_PATH = "/content/drive/MyDrive/BCI_IV_2b"

train_files = sorted(glob.glob(os.path.join(DATA_PATH, "*T.gdf")))
test_files  = sorted(glob.glob(os.path.join(DATA_PATH, "*E.gdf")))

print("Train files:", len(train_files))
print("Test files :", len(test_files))

Train files: 27
Test files : 18


In [ ]:
def process_gdf_file(filepath):
    import numpy as np
    import os
    import mne

    try:
        raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)
    except Exception as e:
        print(f"❌ Error reading {os.path.basename(filepath)}: {e}")
        return None, None

    # -------------------------------------------------
    # 1. Keep only EEG channels (remove EOG leakage)
    # -------------------------------------------------
    raw.pick_types(eeg=True, eog=False)

    # -------------------------------------------------
    # 2. Common Average Reference (VERY IMPORTANT)
    # -------------------------------------------------
    raw.set_eeg_reference('average', projection=False)

    # -------------------------------------------------
    # 3. Bandpass filter for motor imagery
    # -------------------------------------------------
    raw.filter(8., 30., fir_design='firwin', verbose=False)

    # -------------------------------------------------
    # 4. Extract events safely
    # -------------------------------------------------
    try:
        events, event_id = mne.events_from_annotations(raw)
    except Exception as e:
        print(f"⚠️ Event extraction failed in {os.path.basename(filepath)}: {e}")
        return None, None

    if len(events) == 0:
        print(f"⚠️ No events found in {os.path.basename(filepath)}")
        return None, None

    # -------------------------------------------------
    # 5. Robust MI mapping (INDUSTRY SAFE)
    # -------------------------------------------------
    mi_map = {}

    for key, val in event_id.items():
        if key == '769':
            mi_map[val] = 0  # left hand
        elif key == '770':
            mi_map[val] = 1  # right hand

    if len(mi_map) == 0:
        print(f"⚠️ MI events not found in {os.path.basename(filepath)}")
        return None, None

    valid_events = []
    valid_labels = []

    for ev in events:
        code = ev[2]
        if code in mi_map:
            valid_events.append(ev)
            valid_labels.append(mi_map[code])

    if len(valid_events) == 0:
        print(f"⚠️ No valid MI trials in {os.path.basename(filepath)}")
        return None, None

    valid_events = np.array(valid_events)
    valid_labels = np.array(valid_labels)

    # -------------------------------------------------
    # 6. Epoching (BCI IV-2b standard window)
    # -------------------------------------------------
    try:
        epochs = mne.Epochs(
            raw,
            valid_events,
            tmin=0.5,
            tmax=4.5,
            baseline=None,
            preload=True,
            verbose=False
        )
    except Exception as e:
        print(f"⚠️ Epoching failed in {os.path.basename(filepath)}: {e}")
        return None, None

    X = epochs.get_data()  # (trials, channels, samples)
    y = valid_labels[:len(X)]

    # -------------------------------------------------
    # 7. Final safety check
    # -------------------------------------------------
    if X is None or len(X) == 0:
        print(f"⚠️ Empty data after epoching in {os.path.basename(filepath)}")
        return None, None

    return X, y

In [ ]:
print("Train files processed:", len(X_train_list))
print("Test files processed:", len(X_test_list))

Train files processed: 27
Test files processed: 0


In [ ]:
from glob import glob
import os

data_path = "/content/data"

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

gdf_files = sorted(glob(os.path.join(data_path, "*.gdf")))

print("Total GDF files found:", len(gdf_files))

for filepath in gdf_files:

    fname = os.path.basename(filepath)

    X, y = process_gdf_file(filepath)

    if X is None:
        print("Skipped:", fname)
        continue

    # ===== BCI IV-2b naming rule =====
    if "T.gdf" in fname:
        X_train_list.append(X)
        y_train_list.append(y)

    elif "E.gdf" in fname:
        X_test_list.append(X)
        y_test_list.append(y)

    else:
        print("Unknown file type:", fname)

print("\n✅ Train files processed:", len(X_train_list))
print("✅ Test files processed :", len(X_test_list))

Total GDF files found: 45


/tmp/ipykernel_724/2997398702.py:7: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


/tmp/ipykernel_724/2997398702.py:7: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('781')]


/tmp/ipykernel_724/2997398702.py:7: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('781'), np.str_('783')]
⚠️ MI events not found in B0904E.gdf
Skipped: B0904E.gdf


/tmp/ipykernel_724/2997398702.py:7: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('781'), np.str_('783')]
⚠️ MI events not found in B0905E.gdf
Skipped: B0905E.gdf

✅ Train files processed: 27
✅ Test files processed : 0


In [ ]:
X = np.concatenate(X_train_list, axis=0)
y = np.concatenate(y_train_list, axis=0)

print("Combined shape:", X.shape)
print("Labels shape:", y.shape)

Combined shape: (3680, 6, 1001)
Labels shape: (3680,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (2944, 6, 1001)
Test : (736, 6, 1001)


In [ ]:
mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

In [ ]:
X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print(X_train.shape)
print(X_test.shape)

(2944, 6, 1001, 1)
(736, 6, 1001, 1)


In [ ]:
# STEP 7: Combine all processed training files

X = np.concatenate(X_train_list, axis=0)
y = np.concatenate(y_train_list, axis=0)

print("Combined shape:", X.shape)
print("Labels shape:", y.shape)

Combined shape: (3680, 6, 1001)
Labels shape: (3680,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (2944, 6, 1001)
Test : (736, 6, 1001)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (2944, 6, 1001)
Test : (736, 6, 1001)


In [ ]:
# STEP 10: Add CNN channel dimension

X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print("Final train shape:", X_train.shape)
print("Final test shape :", X_test.shape)

Final train shape: (2944, 6, 1001, 1)
Final test shape : (736, 6, 1001, 1)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_eegnet(nb_classes=2, Chans=6, Samples=1001, dropoutRate=0.5):

    input1 = layers.Input(shape=(Chans, Samples, 1))

    # ===== Block 1 =====
    x = layers.Conv2D(
        8, (1, 64),
        padding='same',
        use_bias=False
    )(input1)
    x = layers.BatchNormalization()(x)

    x = layers.DepthwiseConv2D(
        (Chans, 1),
        use_bias=False,
        depth_multiplier=2,
        depthwise_constraint=tf.keras.constraints.max_norm(1.)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 4))(x)
    x = layers.Dropout(dropoutRate)(x)

    # ===== Block 2 =====
    x = layers.SeparableConv2D(
        16, (1, 16),
        use_bias=False,
        padding='same'
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(dropoutRate)(x)

    # ===== Classifier =====
    x = layers.Flatten()(x)
    x = layers.Dense(nb_classes, activation='softmax')(x)

    model = models.Model(inputs=input1, outputs=x)
    return model

model = build_eegnet(
    nb_classes=2,
    Chans=6,
    Samples=1001
)

model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 6, 1001, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 6, 1001, 8)     │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 6, 1001, 8)     │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_8              │ (None, 1, 1001, 16)    │            96 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 1, 1001, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_16 (Activation)      │ (None, 1, 1001, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_16            │ (None, 1, 250, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 1, 250, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_8              │ (None, 1, 250, 16)     │           512 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 1, 250, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_17 (Activation)      │ (None, 1, 250, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_17            │ (None, 1, 31, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 1, 31, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 496)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2)              │           994 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,274 (8.88 KB)

 Trainable params: 2,194 (8.57 KB)

 Non-trainable params: 80 (320.00 B)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled.")

Model compiled.


In [ ]:
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=40,
    validation_split=0.2,
    verbose=1
)

Epoch 1/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 25s 271ms/step - accuracy: 0.4923 - loss: 0.6940 - val_accuracy: 0.4737 - val_loss: 0.6944
Epoch 2/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 17s 227ms/step - accuracy: 0.4948 - loss: 0.6944 - val_accuracy: 0.4737 - val_loss: 0.6937
Epoch 3/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 17s 232ms/step - accuracy: 0.4962 - loss: 0.6940 - val_accuracy: 0.4737 - val_loss: 0.6939
Epoch 4/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 18s 243ms/step - accuracy: 0.4977 - loss: 0.6940 - val_accuracy: 0.4737 - val_loss: 0.6941
Epoch 5/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 17s 225ms/step - accuracy: 0.4888 - loss: 0.6948 - val_accuracy: 0.4737 - val_loss: 0.6934
Epoch 6/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 17s 224ms/step - accuracy: 0.5179 - loss: 0.6929 - val_accuracy: 0.5263 - val_loss: 0.6925
Epoch 7/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 17s 231ms/step - accuracy: 0.5180 - loss: 0.6931 - val_accuracy: 0.5263 - val_loss: 0.6918
Epoch 8/40
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 224ms/step - accuracy: 0.5220 - loss: 0.6917 - val_accu

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ FINAL TEST ACCURACY: {acc*100:.2f}%")


✅ FINAL TEST ACCURACY: 50.00%


In [ ]:
def exponential_standardize(X, eps=1e-4):
    """
    Trial-wise z-score normalization (EEGNet standard practice)
    X shape: (trials, chans, samples, 1)
    """
    X_norm = np.zeros_like(X)

    for i in range(X.shape[0]):
        trial = X[i]
        mean = trial.mean()
        std = trial.std()
        X_norm[i] = (trial - mean) / (std + eps)

    return X_norm

# apply normalization
X_train = exponential_standardize(X_train)
X_test  = exponential_standardize(X_test)

print("✅ Normalization applied")
print("Train mean:", X_train.mean(), "std:", X_train.std())

✅ Normalization applied
Train mean: 4.803397541307451e-22 std: 0.02689669896428767


In [ ]:
model = build_eegnet(
    nb_classes=2,
    Chans=6,
    Samples=1001
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0007),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Fresh model ready")

✅ Fresh model ready


In [ ]:
history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=60,
    validation_split=0.2,
    verbose=1
)

Epoch 1/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 24s 533ms/step - accuracy: 0.4920 - loss: 0.7587 - val_accuracy: 0.5042 - val_loss: 0.6932
Epoch 2/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 17s 447ms/step - accuracy: 0.5148 - loss: 0.7222 - val_accuracy: 0.5008 - val_loss: 0.6929
Epoch 3/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 20s 440ms/step - accuracy: 0.4869 - loss: 0.7280 - val_accuracy: 0.5042 - val_loss: 0.6927
Epoch 4/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 16s 440ms/step - accuracy: 0.5126 - loss: 0.7082 - val_accuracy: 0.5127 - val_loss: 0.6928
Epoch 5/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 21s 442ms/step - accuracy: 0.5198 - loss: 0.7084 - val_accuracy: 0.5195 - val_loss: 0.6920
Epoch 6/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 20s 441ms/step - accuracy: 0.5470 - loss: 0.6950 - val_accuracy: 0.5144 - val_loss: 0.6914
Epoch 7/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 21s 443ms/step - accuracy: 0.5275 - loss: 0.7033 - val_accuracy: 0.5399 - val_loss: 0.6898
Epoch 8/60
37/37 ━━━━━━━━━━━━━━━━━━━━ 21s 458ms/step - accuracy: 0.5042 - loss: 0.7006 - val_accu

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ FINAL TEST ACCURACY: {acc*100:.2f}%")


✅ FINAL TEST ACCURACY: 68.61%


In [25]:
#Fine Tuning to Increase Accuracy

!pip install mne tensorflow scikit-learn

In [26]:
import os
import numpy as np
import mne
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.layers import SeparableConv2D
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [29]:
!unzip "/content/drive/MyDrive/BCI_IV_2b/BCICIV_2b_gdf.zip" -d /content/bci

Archive:  /content/drive/MyDrive/BCI_IV_2b/BCICIV_2b_gdf.zip
  inflating: /content/bci/B0101T.gdf  
  inflating: /content/bci/B0102T.gdf  
  inflating: /content/bci/B0103T.gdf  
  inflating: /content/bci/B0104E.gdf  
  inflating: /content/bci/B0105E.gdf  
  inflating: /content/bci/B0201T.gdf  
  inflating: /content/bci/B0202T.gdf  
  inflating: /content/bci/B0203T.gdf  
  inflating: /content/bci/B0204E.gdf  
  inflating: /content/bci/B0205E.gdf  
  inflating: /content/bci/B0301T.gdf  
  inflating: /content/bci/B0302T.gdf  
  inflating: /content/bci/B0303T.gdf  
  inflating: /content/bci/B0304E.gdf  
  inflating: /content/bci/B0305E.gdf  
  inflating: /content/bci/B0401T.gdf  
  inflating: /content/bci/B0402T.gdf  
  inflating: /content/bci/B0403T.gdf  
  inflating: /content/bci/B0404E.gdf  
  inflating: /content/bci/B0405E.gdf  
  inflating: /content/bci/B0501T.gdf  
  inflating: /content/bci/B0502T.gdf  
  inflating: /content/bci/B0503T.gdf  
  inflating: /content/bci/B0504E.gdf  
  i

In [30]:
data_path = "/content/bci"

gdf_files = []

for root, dirs, files in os.walk(data_path):
    for f in files:
        if f.endswith(".gdf"):
            gdf_files.append(os.path.join(root,f))

print("Total GDF files:",len(gdf_files))

Total GDF files: 45


In [31]:
def process_gdf_file(filepath):

    raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)

    raw.pick_types(eeg=True, eog=False)

    raw.set_eeg_reference('average', projection=False)

    raw.filter(8.,30.,verbose=False)

    events, event_id = mne.events_from_annotations(raw)

    X_list=[]
    y_list=[]

    for key,val in event_id.items():

        if '769' in key:  # left
            label=0
        elif '770' in key: # right
            label=1
        else:
            continue

        selected_events = events[events[:,2]==val]

        if len(selected_events)==0:
            continue

        epochs = mne.Epochs(
            raw,
            selected_events,
            tmin=0.5,
            tmax=4.5,
            baseline=None,
            preload=True,
            verbose=False
        )

        X = epochs.get_data()

        y = np.full(len(X),label)

        X_list.append(X)
        y_list.append(y)

    if len(X_list)==0:
        return None,None

    X=np.concatenate(X_list)
    y=np.concatenate(y_list)

    return X,y

In [32]:
X_list=[]
y_list=[]

for file in gdf_files:

    X,y = process_gdf_file(file)

    if X is not None:
        X_list.append(X)
        y_list.append(y)

print("Files with trials:",len(X_list))

/tmp/ipykernel_661/2065324162.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('781'), np.str_('783')]


/tmp/ipykernel_661/2065324162.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('781'), np.str_('783')]


/tmp/ipykernel_661/2065324162.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


/tmp/ipykernel_661/2065324162.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


/tmp/ipykernel_661/2065324162.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('781')]
Files with trials: 27


In [33]:
X = np.concatenate(X_list)
y = np.concatenate(y_list)

print("Dataset shape:",X.shape)
print("Labels shape:",y.shape)

Dataset shape: (3680, 6, 1001)
Labels shape: (3680,)


In [34]:
X = (X - np.mean(X,axis=2,keepdims=True)) / np.std(X,axis=2,keepdims=True)

In [35]:
X = X[:,:,:,np.newaxis]

print("New shape:",X.shape)

New shape: (3680, 6, 1001, 1)


In [36]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:",X_train.shape)
print("Test:",X_test.shape)

Train: (2944, 6, 1001, 1)
Test: (736, 6, 1001, 1)


In [37]:
def EEGNet(nb_classes=2,Chans=6,Samples=1001):

    input1 = Input(shape=(Chans,Samples,1))

    block1 = Conv2D(16,(1,64),padding='same',use_bias=False)(input1)
    block1 = BatchNormalization()(block1)

    block1 = DepthwiseConv2D(
        (Chans,1),
        use_bias=False,
        depth_multiplier=2,
        depthwise_constraint=max_norm(1.)
    )(block1)

    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)

    block1 = AveragePooling2D((1,4))(block1)
    block1 = Dropout(0.5)(block1)

    block2 = SeparableConv2D(
        32,(1,16),
        use_bias=False,
        padding='same'
    )(block1)

    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)

    block2 = AveragePooling2D((1,8))(block2)
    block2 = Dropout(0.5)(block2)

    flatten = Flatten()(block2)

    dense = tf.keras.layers.Dense(
        nb_classes,
        kernel_constraint=max_norm(0.25)
    )(flatten)

    softmax = Activation('softmax')(dense)

    return Model(inputs=input1,outputs=softmax)

In [38]:
model = EEGNet()

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(0.001),
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6, 1001, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 6, 1001, 16)    │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 6, 1001, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 1, 1001, 32)    │           192 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 1001, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1, 1001, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 1, 250, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 250, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 1, 250, 32)     │         1,536 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 250, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1, 250, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 1, 31, 32)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 31, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 992)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         1,986 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,058 (19.76 KB)

 Trainable params: 4,898 (19.13 KB)

 Non-trainable params: 160 (640.00 B)

In [39]:
history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=60,
    validation_data=(X_test,y_test),
    verbose=1
)

Epoch 1/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 37s 349ms/step - accuracy: 0.5106 - loss: 0.7028 - val_accuracy: 0.5000 - val_loss: 0.6930
Epoch 2/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 28s 299ms/step - accuracy: 0.5163 - loss: 0.6939 - val_accuracy: 0.5217 - val_loss: 0.6898
Epoch 3/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 29s 312ms/step - accuracy: 0.5880 - loss: 0.6585 - val_accuracy: 0.6277 - val_loss: 0.6449
Epoch 4/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 35s 376ms/step - accuracy: 0.6272 - loss: 0.6341 - val_accuracy: 0.6658 - val_loss: 0.6074
Epoch 5/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 34s 303ms/step - accuracy: 0.6432 - loss: 0.6214 - val_accuracy: 0.6957 - val_loss: 0.5863
Epoch 6/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 28s 300ms/step - accuracy: 0.6646 - loss: 0.6062 - val_accuracy: 0.6793 - val_loss: 0.5841
Epoch 7/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 28s 305ms/step - accuracy: 0.6429 - loss: 0.6135 - val_accuracy: 0.6875 - val_loss: 0.5742
Epoch 8/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 38s 276ms/step - accuracy: 0.6442 - loss: 0.6195 - val_accu

In [40]:
loss,acc = model.evaluate(X_test,y_test,verbose=0)

print("Final Accuracy:",acc*100)

Final Accuracy: 73.097825050354


In [41]:
y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred,axis=1)

print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step
[[296  72]
 [126 242]]
              precision    recall  f1-score   support

           0       0.70      0.80      0.75       368
           1       0.77      0.66      0.71       368

    accuracy                           0.73       736
   macro avg       0.74      0.73      0.73       736
weighted avg       0.74      0.73      0.73       736



In [10]:
#FilterBankCSP + EEGNet Hybrid

!pip install mne tensorflow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 79.7 MB/s eta 0:00:00


In [11]:
import os
import glob
import numpy as np
import mne

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!unzip "/content/drive/MyDrive/BCI_IV_2b/BCICIV_2b_gdf.zip" -d /content/bci

Archive:  /content/drive/MyDrive/BCI_IV_2b/BCICIV_2b_gdf.zip
  inflating: /content/bci/B0101T.gdf  
  inflating: /content/bci/B0102T.gdf  
  inflating: /content/bci/B0103T.gdf  
  inflating: /content/bci/B0104E.gdf  
  inflating: /content/bci/B0105E.gdf  
  inflating: /content/bci/B0201T.gdf  
  inflating: /content/bci/B0202T.gdf  
  inflating: /content/bci/B0203T.gdf  
  inflating: /content/bci/B0204E.gdf  
  inflating: /content/bci/B0205E.gdf  
  inflating: /content/bci/B0301T.gdf  
  inflating: /content/bci/B0302T.gdf  
  inflating: /content/bci/B0303T.gdf  
  inflating: /content/bci/B0304E.gdf  
  inflating: /content/bci/B0305E.gdf  
  inflating: /content/bci/B0401T.gdf  
  inflating: /content/bci/B0402T.gdf  
  inflating: /content/bci/B0403T.gdf  
  inflating: /content/bci/B0404E.gdf  
  inflating: /content/bci/B0405E.gdf  
  inflating: /content/bci/B0501T.gdf  
  inflating: /content/bci/B0502T.gdf  
  inflating: /content/bci/B0503T.gdf  
  inflating: /content/bci/B0504E.gdf  
  i

In [12]:
data_path = "/content/bci"

In [13]:
all_files = sorted(glob.glob(os.path.join(data_path,"*.gdf")))

train_files = [f for f in all_files if "T.gdf" in f]

print("Total files:",len(all_files))
print("Training files:",len(train_files))

Total files: 45
Training files: 27


In [14]:
filter_bands = [
    (8,12),
    (12,16),
    (16,20),
    (20,24),
    (24,28),
    (28,30)
]

In [15]:
def process_gdf_filterbank(filepath):

    raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)

    raw.pick("eeg")

    raw.set_eeg_reference('average')

    raw.filter(0.5,40,verbose=False)

    # Extract events
    events, event_id = mne.events_from_annotations(raw)

    # Ensure MI events exist
    if '769' not in event_id or '770' not in event_id:
        return None, None

    left_event = event_id['769']
    right_event = event_id['770']

    # Select only MI trials
    mask = np.logical_or(events[:,2]==left_event, events[:,2]==right_event)

    events = events[mask]

    if len(events)==0:
        return None,None

    labels = np.where(events[:,2]==left_event,0,1)

    band_trials=[]

    for f_low,f_high in filter_bands:

        raw_band = raw.copy()
        raw_band.filter(f_low,f_high,verbose=False)

        epochs = mne.Epochs(
            raw_band,
            events,
            tmin=0.5,
            tmax=4.5,
            baseline=None,
            preload=True,
            verbose=False
        )

        X = epochs.get_data()

        band_trials.append(X)

    X = np.concatenate(band_trials,axis=1)

    return X,labels

In [16]:
X_list=[]
y_list=[]

processed=0

for file in train_files:

    X,y = process_gdf_filterbank(file)

    if X is not None and len(X)>0:

        X_list.append(X)
        y_list.append(y)

        processed+=1

print("Files processed:",processed)

/tmp/ipykernel_446/544310683.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


/tmp/ipykernel_446/544310683.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


/tmp/ipykernel_446/544310683.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


...

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


/tmp/ipykernel_446/544310683.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]


/tmp/ipykernel_446/544310683.py:3: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filepath, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('781')]
Files processed: 27


In [17]:
if len(X_list)==0:
    raise ValueError("No data extracted from files")

X = np.concatenate(X_list)
y = np.concatenate(y_list)

print("Dataset shape:",X.shape)

Dataset shape: (3680, 36, 1001)


In [18]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [19]:
X_train = X_train[...,np.newaxis]
X_test  = X_test[...,np.newaxis]

print(X_train.shape)
print(X_test.shape)

(2944, 36, 1001, 1)
(736, 36, 1001, 1)


In [20]:
from mne.decoding import CSP

csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

X_csp = csp.fit_transform(X, y)

print(X_csp.shape)

Computing rank from data with rank=None
    Using tolerance 6.4e-05 (2.2e-16 eps * 36 dim * 8e+09  max singular value)
    Estimated rank (data): 33
    data: rank 33 computed from 36 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 36 -> 33
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
(3680, 4)


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_reshaped = X.reshape(X.shape[0], -1)

X_scaled = scaler.fit_transform(X_reshaped)

X = X_scaled.reshape(X.shape)

print(X.shape)

(3680, 36, 1001)


In [22]:
X = X[:,:,:,np.newaxis]

print(X.shape)

(3680, 36, 1001, 1)


In [23]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import AveragePooling2D, Dropout
from tensorflow.keras.layers import SeparableConv2D, Flatten, Dense
from tensorflow.keras.layers import BatchNormalization, Activation

In [26]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import AveragePooling2D, Dropout
from tensorflow.keras.layers import SeparableConv2D, Flatten, Dense
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.constraints import max_norm

def EEGNet(nb_classes=2,Chans=6,Samples=1001):

    input1 = Input(shape=(Chans,Samples,1))

    block1 = Conv2D(16,(1,64),padding='same',use_bias=False)(input1)
    block1 = BatchNormalization()(block1)

    block1 = DepthwiseConv2D(
        (Chans,1),
        use_bias=False,
        depth_multiplier=2,
        depthwise_constraint=max_norm(1.)
    )(block1)

    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)

    block1 = AveragePooling2D((1,4))(block1)
    block1 = Dropout(0.5)(block1)

    block2 = SeparableConv2D(
        32,(1,16),
        use_bias=False,
        padding='same'
    )(block1)

    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)

    block2 = AveragePooling2D((1,8))(block2)
    block2 = Dropout(0.5)(block2)

    flatten = Flatten()(block2)

    dense = tf.keras.layers.Dense(
        nb_classes,
        kernel_constraint=max_norm(0.25)
    )(flatten)

    softmax = Activation('softmax')(dense)

    return Model(inputs=input1,outputs=softmax)

model = EEGNet(
    nb_classes=2,
    Chans=X_train.shape[1],
    Samples=X_train.shape[2]
)

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(0.001),
    metrics=['accuracy']
)

model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test,y_test)
)

Epoch 1/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 40ms/step - accuracy: 0.5096 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5058 - loss: 0.6934 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 3/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5119 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 4/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5141 - loss: 0.6925 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 5/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.4931 - loss: 0.6937 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 6/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5078 - loss: 0.6936 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 7/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.4906 - loss: 0.6938 - val_accuracy: 0.5000 - val_loss: 0.6945
Epoch 8/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.4789 - loss: 0.6939 - val_accuracy: 0

In [28]:
loss,acc = model.evaluate(X_test,y_test)

print("EEGNet Accuracy:",acc*100)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5286 - loss: 0.8799
EEGNet Accuracy: 51.902174949645996
